# Permute Operations in PyTorch

### Understanding dimensions

In [1]:
import torch

In [2]:
x1 = torch.randn(2, 3, 4)
x1

tensor([[[ 0.5139,  0.0022, -1.3745,  1.0908],
         [ 0.2581, -1.1080,  1.2592, -1.5518],
         [-1.2215, -0.5456, -0.3176, -1.4390]],

        [[-1.6980,  1.2136,  0.0642,  0.6134],
         [-0.6072, -0.4743,  1.1755,  0.5754],
         [ 0.1881,  0.9700,  0.1947,  1.7852]]])

In [3]:
x1.shape

torch.Size([2, 3, 4])

Dimension 0 → size 2  
Dimension 1 → size 3  
Dimension 2 → size 4  

In [8]:
x1_permuted = x1.permute(1, 0, 2)
x1_permuted

tensor([[[ 0.5139,  0.0022, -1.3745,  1.0908],
         [-1.6980,  1.2136,  0.0642,  0.6134]],

        [[ 0.2581, -1.1080,  1.2592, -1.5518],
         [-0.6072, -0.4743,  1.1755,  0.5754]],

        [[-1.2215, -0.5456, -0.3176, -1.4390],
         [ 0.1881,  0.9700,  0.1947,  1.7852]]])

In [7]:
x1_permuted.shape

torch.Size([3, 2, 4])

New dimension 0 ← Old dimension 1  
New dimension 1 ← Old dimension 0  
New dimension 2 ← Old dimension 2  

### Visual Example

In [9]:
x2 = torch.arange(24).reshape(2, 3, 4)
x2 # batch

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])

In [11]:
x2[0]

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [12]:
x2_permuted = x2.permute(1,0,2)
x2_permuted

tensor([[[ 0,  1,  2,  3],
         [12, 13, 14, 15]],

        [[ 4,  5,  6,  7],
         [16, 17, 18, 19]],

        [[ 8,  9, 10, 11],
         [20, 21, 22, 23]]])

### What is `torch.permute()`?

`torch.permute()` is one of the most important tensor operations in PyTorch. It **reorders the dimensions (axes)** of a tensor without changing its underlying data. It returns a **view** of the original tensor, meaning no new memory is allocated (unless you later make it contiguous).

### Syntax

`tensor.permute(*dims)` or `torch.permute(tensor, dims)`

- `dims` is a tuple specifying the **new order of dimensions**.
- Every dimension must appear **exactly once**.

### Example with Images

Many deep learning models expect channel-first format `(C, H, W)` but images often come as `(H, W, C)`. `permute` converts between them:

```python
# (H, W, C) → (C, H, W)
image = image.permute(2, 0, 1)
```

**Meaning:** New dim 0 ← Old dim 2 (Channels), New dim 1 ← Old dim 0 (Height), New dim 2 ← Old dim 1 (Width).

A tensor of shape `(224, 224, 3)` becomes `(3, 224, 224)`.

Common conversions:

| From | To | permute |
|------|----|---------|
| HWC | CHW | `permute(2, 0, 1)` |
| NCHW | NHWC | `permute(0, 2, 3, 1)` |
| NHWC | NCHW | `permute(0, 3, 1, 2)` |
| THWC | CTHW | `permute(3, 0, 1, 2)` |

In [13]:
# Example: (H, W, C) -> (C, H, W) as used in image processing
hwc = torch.randn(224, 224, 3)
chw = hwc.permute(2, 0, 1)
print("HWC shape:", hwc.shape)
print("CHW shape:", chw.shape)

# Example: (N, H, W, C) -> (N, C, H, W)
nhwc = torch.randn(8, 224, 224, 3)
nchw = nhwc.permute(0, 3, 1, 2)
print("NHWC shape:", nhwc.shape)
print("NCHW shape:", nchw.shape)

HWC shape: torch.Size([224, 224, 3])
CHW shape: torch.Size([3, 224, 224])
NHWC shape: torch.Size([8, 224, 224, 3])
NCHW shape: torch.Size([8, 3, 224, 224])


### `permute` vs `transpose`

| Feature | `permute` | `transpose` |
|---------|-----------|-------------|
| Number of dimensions | Any number | Exactly two |
| Flexibility | High — full reorder | Simple — swap only |
| Example | `permute(2, 0, 1)` | `transpose(0, 1)` |

For a tensor of shape `(2, 3, 4)`:
- `transpose(0, 1)` → `(3, 2, 4)`
- `permute(2, 1, 0)` → `(4, 3, 2)`

In [14]:
# permute vs transpose comparison
x = torch.randn(2, 3, 4)

# transpose — swaps two specific axes
t = x.transpose(0, 1)
print("transpose(0, 1):", t.shape)

# permute — arbitrary reorder
p = x.permute(2, 1, 0)
print("permute(2, 1, 0):", p.shape)

transpose(0, 1): torch.Size([3, 2, 4])
permute(2, 1, 0): torch.Size([4, 3, 2])


### `permute` vs `reshape`

These operations serve very different purposes.

| | `permute` | `reshape` |
|---|-----------|-----------|
| What it does | Reorders dimensions | Changes the tensor's shape |
| Element order | Preserved | Rearranged |
| Memory | Creates a view | May view or copy |
| Example | `permute(2, 1, 0)` from `(2,3,4)` → `(4,3,2)` | `reshape(6, 4)` from `(2,3,4)` → `(6,4)` |

`permute` changes **which axis comes first**, while `reshape` changes **how the same elements are organized into a new shape**.

In [15]:
# permute vs reshape comparison
x = torch.arange(24).reshape(2, 3, 4)
print("Original shape:", x.shape)

# permute — reorders axes
p = x.permute(2, 1, 0)
print("permute(2, 1, 0) shape:", p.shape)

# reshape — reorganises elements into new shape
r = x.reshape(6, 4)
print("reshape(6, 4) shape:", r.shape)

Original shape: torch.Size([2, 3, 4])
permute(2, 1, 0) shape: torch.Size([4, 3, 2])
reshape(6, 4) shape: torch.Size([6, 4])


### `contiguous()` — Why it's sometimes needed

After `permute()`, tensor elements are accessed in a different order, but the underlying memory layout hasn't changed. Some operations (like `.view()`) require the tensor to be stored contiguously in memory.

```python
y = x.permute(1, 0)
y.is_contiguous()   # False
y = y.contiguous()
y.is_contiguous()   # True
```

In [16]:
# contiguous example
x = torch.randn(2, 3, 4)
y = x.permute(1, 0, 2)

print("is contiguous after permute:", y.is_contiguous())

# .view() would fail here — need contiguous first
y_contiguous = y.contiguous()
print("is contiguous after .contiguous():", y_contiguous.is_contiguous())

# Now .view() works
y_reshaped = y_contiguous.view(3, -1)
print("reshaped view:", y_reshaped.shape)

is contiguous after permute: False
is contiguous after .contiguous(): True
reshaped view: torch.Size([3, 8])


### Real-World Example 1: Images (HWC ↔ CHW)

When you load an image with libraries like PIL or OpenCV, the shape is usually `(H, W, C)` — height, width, channels. PyTorch models (CNNs, ViTs) expect `(C, H, W)`, so you `permute` the axes.

In [17]:
# Simulate loading an image: (H, W, C) -> (C, H, W)
import torch

# Pretend we loaded a 224x224 RGB image via PIL (H, W, C)
img_hwc = torch.randn(224, 224, 3)
print(f"Loaded image shape (H, W, C): {img_hwc.shape}")

# Permute to channel-first for PyTorch models
img_chw = img_hwc.permute(2, 0, 1)
print(f"After permute(2, 0, 1) -> (C, H, W): {img_chw.shape}")

# Visual proof: element access stays the same
assert img_hwc[100, 50, 1] == img_chw[1, 100, 50]
print("✓ Element access preserved: img_hwc[100, 50, 1] == img_chw[1, 100, 50]")

Loaded image shape (H, W, C): torch.Size([224, 224, 3])
After permute(2, 0, 1) -> (C, H, W): torch.Size([3, 224, 224])
✓ Element access preserved: img_hwc[100, 50, 1] == img_chw[1, 100, 50]


### Real-World Example 2: Batch of Images (NHWC ↔ NCHW)

When you stack multiple images into a batch, the shape is `(N, H, W, C)`. Most PyTorch vision models expect `(N, C, H, W)`.

In [18]:
# Batch of images: (N, H, W, C) -> (N, C, H, W)
batch_nhwc = torch.randn(8, 224, 224, 3)
print(f"Batch NHWC: {batch_nhwc.shape}")

batch_nchw = batch_nhwc.permute(0, 3, 1, 2)
print(f"After permute(0, 3, 1, 2) -> NCHW: {batch_nchw.shape}")

# Going back: NCHW -> NHWC (e.g. after model output, for visualization)
back_to_nhwc = batch_nchw.permute(0, 2, 3, 1)
print(f"Back to NHWC: {back_to_nhwc.shape}")

Batch NHWC: torch.Size([8, 224, 224, 3])
After permute(0, 3, 1, 2) -> NCHW: torch.Size([8, 3, 224, 224])
Back to NHWC: torch.Size([8, 224, 224, 3])


### Real-World Example 3: Video (T H W C → C T H W)

A video is a sequence of frames, typically shaped `(T, H, W, C)` where T is the number of frames. Video models often want channels first: `(C, T, H, W)`.

In [19]:
# Simulate a video clip: 32 frames, 224x224, RGB
video_thwc = torch.randn(32, 224, 224, 3)
print(f"Video (T, H, W, C): {video_thwc.shape}")

# Permute to (C, T, H, W) for video models like I3D, ViViT
video_cthw = video_thwc.permute(3, 0, 1, 2)
print(f"After permute(3, 0, 1, 2) -> (C, T, H, W): {video_cthw.shape}")

# Also common: (N, T, H, W, C) -> (N, C, T, H, W) for batched video
batch_video = torch.randn(4, 32, 224, 224, 3)
batch_video_permuted = batch_video.permute(0, 4, 1, 2, 3)
print(f"Batched video: {batch_video.shape} -> {batch_video_permuted.shape}")

Video (T, H, W, C): torch.Size([32, 224, 224, 3])
After permute(3, 0, 1, 2) -> (C, T, H, W): torch.Size([3, 32, 224, 224])
Batched video: torch.Size([4, 32, 224, 224, 3]) -> torch.Size([4, 3, 32, 224, 224])


### Real-World Example 4: NLP / Sequence Data

Transformers process sequences shaped `(batch, seq_len, features)`. Sometimes you need to reorder axes, e.g. when working with multi-head attention where you want `(batch, num_heads, seq_len, head_dim)`.

In [20]:
# NLP: (batch, seq_len, features) -> reorder for multi-head attention
batch = 4
seq_len = 128
features = 768
num_heads = 12
head_dim = features // num_heads  # 64

# Standard transformer hidden state
hidden = torch.randn(batch, seq_len, features)
print(f"Transformer output (B, S, F): {hidden.shape}")

# Reshape + permute to get (B, num_heads, S, head_dim) for attention
hidden_reshaped = hidden.view(batch, seq_len, num_heads, head_dim)
hidden_mha = hidden_reshaped.permute(0, 2, 1, 3)
print(f"For multi-head attention (B, H, S, D): {hidden_mha.shape}")

# Permute back after attention
hidden_back = hidden_mha.permute(0, 2, 1, 3).reshape(batch, seq_len, features)
print(f"Permuted back (B, S, F): {hidden_back.shape}")

Transformer output (B, S, F): torch.Size([4, 128, 768])
For multi-head attention (B, H, S, D): torch.Size([4, 12, 128, 64])
Permuted back (B, S, F): torch.Size([4, 128, 768])


### Mental Model

If a tensor has dimensions `(A, B, C, D)`, then:

```
permute(2, 0, 3, 1)
```

means:

```
New dim 0 = Old dim 2 (C)
New dim 1 = Old dim 0 (A)
New dim 2 = Old dim 3 (D)
New dim 3 = Old dim 1 (B)
```

Resulting shape: `(C, A, D, B)`

A simple rule: **The numbers tell PyTorch which original dimension goes to each new position.** Each index must appear exactly once — you're only changing the *order of axes*, never the data.